وقتی داده ها فیلتر شدند این برنامه نمودار تولید بر اساس دما را برای تمام واحدهای تولیدی رسم کرده و داده های مربوط به آخرین سری تولید و داده های انتخاب شده توسط فیلتر پنجم را نیز مشخص میکند. 

In [80]:
import os
import sys

import pandas as pd
import plotly.graph_objects as go

import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor, LinearRegression
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
from src.models.filter_data.filter_data import *
from src.models.filter_data.feature_adder import *

In [81]:
csv_read_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df_c = pd.read_csv(csv_read_path, encoding='utf-8')

In [97]:
def get_Xy(df, name, code, temp_feature):
    ds = Data_selector(Data_selector(df).select_peaks(goodness=5))
    df_plot = ds.filter_name_code(name, code)
    try:
        features = ["generation", f"{temp_feature}"]
        df_modified = df_plot[features].copy(deep=True)
    
        sens_temps = df_modified[temp_feature].values
        gens = df_modified['generation'].values
    
        X, y = find_points_on_envelope(gens, sens_temps, bin_length=100, p=0.95)
        # print(name, code, int(len(df_plot) / (365 * 24) * 10) / 10)
        return X.flatten(), y
    except Exception as e:
        print(name, code, "None")
        print(e)
        return None,None


In [98]:
def get_dict_by_feature(df_c,temp_feature):
    power_plants = df_c[['name', 'code']].drop_duplicates()
    Xy_by_name_code = {}
    for row in power_plants.itertuples():
        name, code = row.name, row.code
        X,y = get_Xy(df_c, name, code,temp_feature)
        Xy_by_name_code[(name, code)] = X,y
    return Xy_by_name_code

In [99]:
def draw_plot(X, y, t_c, fig, name, code):
    
    X = X.flatten()
    y_i = y[X > t_c]
    X_i = X[X > t_c]
    
    fig.add_trace(
        go.Scatter(
            x=X_i,
            y=y_i,
            #mode='markers',
            mode="lines",
            name=f"set {name}-{code}"
        )
    )

In [100]:
from src.models import utils
def err_line(X,y,model,th):
    model.fit(X.reshape(-1,1),y)
    y_i_pred = model.predict(X.reshape(-1,1))
    err = utils.compute_relative_rmse(y_i_pred,y)
    return err

In [101]:
names = ['hour', 'forecasted_load',
       'required', 'declared', 'status', 'temp_sens', 'scadaf',
       'temperature', 'humidity', 'dew', 'apparent_temperature',
       'precipitation', 'rain', 'snow', 'surface_pressure',
       'evapotranspiration', 'wind_speed', 'wind_direction',
        'season', 'day_of_week', 'month']

In [109]:
import plotly.graph_objects as go

fig = go.Figure()
t_c = 0
th = 2
t = 0
errors = []
kk = 20
h = np.array([1]*(2*kk+1))/(2*kk+1)
temp_feature = "dew"

Xy_by_name_code = get_dict_by_feature(df_c,temp_feature)
power_plants = df_c[['name', 'code']].drop_duplicates()
for row in power_plants.itertuples():
    name, code = row.name, row.code
    X,y = Xy_by_name_code[(name, code)]
    if len(y) < len(h) : continue
    y = np.convolve(y, h, mode="valid")
    X = np.array(X)[kk:-kk]
    y = (y / y.mean() - 1)*100
    err = err_line(X,y,model=LinearRegression(),th=th)
    if 1 < abs(err) or 1==1:   
        draw_plot(X, y, t_c, fig, name, code)
        t +=1
    errors.append(err)
fig.update_layout(
    title="Multiple XY Sets",
    xaxis_title=temp_feature,
    yaxis_title="Generation"
)
fig.write_html(
            f"{project_root}/src/visualization/unit_figs/test_features/ALL_{temp_feature}.html")
# _ = plt.hist(errors,bins = 1000)

In [ ]:
import plotly.graph_objects as go
from collections import defaultdict

# یک دیکشنری برای جمع کردن داده‌های هر name
plots = defaultdict(list)

# جدا کردن داده‌های هر name
for row in power_plants.itertuples():
    name, code = row.name, row.code
    X, y = Xy_by_name_code[(name, code)]
    plots[name].append((code, X, y))

# برای هر name یک نمودار جدا بساز
for name, items in plots.items():
    fig = go.Figure()

    for code, X, y in items:
        fig.add_trace(
            go.Scatter(
                x=X,
                y=y,
                mode="markers",
                name=f"{name}-{code}"
            )
        )

    fig.update_layout(
        title=f"{name} - Units",
        xaxis_title="Time",
        yaxis_title="Generation"
    )

    fig.write_html(
            f"{project_root}/src/visualization/unit_figs/filter5/{name}.html"
        )
